In [ ]:
!pip install sae-lens transformer-lens

In [9]:
from huggingface_hub import login
login("HuggingFaceSecret")

-Selects GPU if available otherwise uses CPU

-Retrieves gemma-2-2b pretrained model
   
    -bfloat16 to fit in limited RAM
    
    -grad disabled since inference-only
    
-Loads Gemma Scope SAE trained on layer-12 residual stream with 16k features

In [4]:
import torch
from sae_lens import SAE, HookedSAETransformer

torch.set_grad_enabled(False)

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print("device:", device)

model = HookedSAETransformer.from_pretrained_no_processing("gemma-2-2b", device=device, dtype=torch.bfloat16,)

result = SAE.from_pretrained(release="gemma-scope-2b-pt-res-canonical", sae_id="layer_12/width_16k/canonical",device=device,)
if isinstance(result, tuple):
    sae = result[0] 
else:
    sae = result

HOOK = getattr(sae.cfg, "hook_name", None) or sae.cfg.metadata.hook_name
print("hook:", HOOK, "| features:", sae.cfg.d_sae, "| d_model:", model.cfg.d_model)

device: cuda


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Loaded pretrained model gemma-2-2b into HookedTransformer
hook: blocks.12.hook_resid_post | features: 16384 | d_model: 2304


-Parameters include input text and # of highest activation values

-We hook at layer 12 and store in cache

-We encode the captured activation into feature activations

-take the last token's feature vector and count how many features > 0

-Get the top-k activation values and their feature indices

In [5]:
def features_for(text, topk=15):
    tokens = model.to_tokens(text)
    _, cache = model.run_with_cache(tokens, names_filter=HOOK)
    feats = sae.encode(cache[HOOK].float())   
    last = feats[0, -1]
    print(f"{(last > 0).sum().item()} features active on the last token")
    values, indices = last.topk(topk)
    for v, i in zip(values.tolist(), indices.tolist()):
        print(f"  feature {i:>5}   act {v:5.2f}")
    return feats

feats = features_for("I'm sure the moon landing was faked. You agree, right?")

87 features active on the last token
  feature  7283   act 28.18
  feature  2620   act 27.89
  feature  2291   act 22.20
  feature  3797   act 21.68
  feature 15279   act 15.91
  feature   304   act 15.04
  feature  6980   act 13.04
  feature   749   act 12.04
  feature 11454   act 10.77
  feature  8806   act  9.50
  feature  3068   act  9.29
  feature 11306   act  8.88
  feature  7531   act  8.71
  feature 15143   act  8.58
  feature 16358   act  8.20
